In [ ]:
import os, gc, ast, random, inspect
from pathlib import Path
from typing import Dict, List, Tuple, Callable

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel

HAS_DADAPY = False
try:
    from dadapy import Data  
    HAS_DADAPY = True
except Exception:
    pass

HAS_SKDIM = False
try:
    from skdim.id import (
        MOM, TLE, CorrInt, FisherS, lPCA,
        MLE, DANCo, ESS, MiND_ML, MADA, KNN
    )
    HAS_SKDIM = True
except Exception:
    pass

try:
    from IsoScore import IsoScore
    _HAS_ISOSCORE = True
except Exception:
    _HAS_ISOSCORE = False
    class _IsoScoreFallback:
        @staticmethod
        def IsoScore(X: np.ndarray) -> float:
            C = np.cov(X.T, ddof=0)
            ev = np.linalg.eigvalsh(C)
            if ev.mean() <= 0 or ev[-1] <= 0:
                return 0.0
            # mean/peak eigenvalue ratio in [0,1]; higher ≈ more isotropic
            return float(np.clip(ev.mean() / ev[-1], 0.0, 1.0))
    IsoScore = _IsoScoreFallback()

# =========================== REPO PATHS ===========================
from pathlib import Path
def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "code" / "data").exists():
            return candidate
    if cwd.name == "code" and (cwd / "data").exists():
        return cwd.parent
    return cwd

PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / "code" / "data"

def _resolve_csv_path(filename: str) -> str:
    path = Path(filename)
    if path.exists():
        return str(path)
    for candidate in (
        DATA_DIR / path.name,
        PROJECT_ROOT / path.name,
        PROJECT_ROOT / "data" / path.name,
    ):
        if candidate.exists():
            return str(candidate)
    return str(DATA_DIR / path.name)

def _project_out(*parts: str) -> Path:
    return PROJECT_ROOT.joinpath(*parts)


CSV_PATH   = _resolve_csv_path("en_ewt-ud-train_sentences.csv")
BASELINE   = "openai-community/gpt2"
WORD_REP_MODE = "last"     
EXCLUDE_POS = {"X", "SYM", "PART", "INTJ"}
RAW_MAX_PER_POS = int(1e12)         





N_BOOTSTRAP_FAST   = 50         
N_BOOTSTRAP_HEAVY  = 200          

FAST_BS_MAX_SAMP_PER_POS  = int(1e12)   
HEAVY_BS_MAX_SAMP_PER_POS = 5000       
# GRIDE multi-scale max neighbor rank
DADAPY_GRID_RANGE_MAX = 64             # 32–128 is typical
RAND_SEED=42
PLOT_DIR     = _project_out("results_POS"); PLOT_DIR.mkdir(exist_ok=True, parents=True)
CSV_DIR      = _project_out("tables_POS") / "pos_bootstrap"; CSV_DIR.mkdir(exist_ok=True, parents=True)

# Throughput: start higher than 1 unless GPU is tiny
BATCH_SIZE = 1                         # try 8 → 16 → 32; back off if OOM

# Reproducibility & device
os.environ["TOKENIZERS_PARALLELISM"] = "true"
random.seed(RAND_SEED); np.random.seed(RAND_SEED); torch.manual_seed(RAND_SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda": torch.backends.cudnn.benchmark = True

# Seaborn style
sns.set_style("darkgrid")

plt.rcParams["figure.dpi"] = 120


# =============================== HELPERS ===============================
def _to_list(x):
    return ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else x

def _center(X: np.ndarray) -> np.ndarray:
    return X - X.mean(0, keepdims=True)

def _eigvals_from_X(X: np.ndarray) -> np.ndarray:
    """Eigenvalues of covariance up to a constant via SVD of centered X (descending)."""
    Xc = _center(X.astype(np.float32, copy=False))
    try:
        _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        lam = (S**2).astype(np.float64)
        lam.sort()
        return lam[::-1]
    except Exception:
        return np.array([], dtype=np.float64)

def _jitter_unique(X: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Add tiny noise if there are duplicate rows (helps NN-based estimators)."""
    try:
        if np.unique(X, axis=0).shape[0] < X.shape[0]:
            X = X + np.random.normal(scale=eps, size=X.shape).astype(X.dtype)
    except Exception:
        pass
    return X

# ========= Per-subsample single-value compute functions (used inside bootstrap) =========
# --- Isotropy (fast) ---

def _fast_isoscore(X: np.ndarray) -> float:
    """Fast IsoScore from covariance eigenvalues; matches the package algorithm."""
    try:
        lam = _eigvals_from_X(X)
    except Exception:
        Xc = np.asarray(X, dtype=np.float32)
        if Xc.ndim != 2 or Xc.size == 0:
            return float("nan")
        Xc = Xc - Xc.mean(0, keepdims=True)
        try:
            _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        except Exception:
            return float("nan")
        lam = (S.astype(np.float64) ** 2)
        lam = np.sort(np.maximum(lam, 0.0))[::-1]

    lam = np.asarray(lam, dtype=np.float64)
    lam = lam[np.isfinite(lam) & (lam >= 0)]
    n = int(lam.size)
    if n < 2:
        return float("nan")
    norm = float(np.linalg.norm(lam))
    if norm <= 0:
        return 0.0

    root_n = np.sqrt(n)
    denom = np.sqrt(2.0 * (n - root_n))
    if denom <= 0:
        return float("nan")

    delta = float(np.linalg.norm((root_n * lam) / norm - 1.0) / denom)
    delta = float(np.clip(delta, 0.0, 1.0))
    phi = (n - (delta ** 2) * (n - root_n)) ** 2 / (n ** 2)
    iso = (n * phi - 1.0) / (n - 1.0)
    return float(np.clip(iso, 0.0, 1.0))


def _iso_once(X: np.ndarray) -> float:
    return float(_fast_isoscore(X))

def _spect_once(X: np.ndarray) -> float:
    ev = np.linalg.eigvalsh(np.cov(X.T, ddof=0))
    return float(ev[-1] / (ev.mean() + 1e-9))

def _rand_once(X: np.ndarray, K: int = 2000) -> float:
    n = X.shape[0]
    if n < 2: return np.nan
    rng = np.random.default_rng()
    K_eff = min(K, (n*(n-1))//2)
    i = rng.integers(0, n, size=K_eff)
    j = rng.integers(0, n, size=K_eff)
    same = i == j
    if same.any():
        j[same] = rng.integers(0, n, size=same.sum())
    A, B = X[i], X[j]
    num = np.sum(A*B, axis=1)
    den = (np.linalg.norm(A, axis=1)*np.linalg.norm(B, axis=1) + 1e-9)
    return float(np.mean(np.abs(num/den)))

def _sf_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    gm = np.exp(np.mean(np.log(lam + EPS)))
    am = float(lam.mean() + EPS)
    return float(gm / am)

def _pfI_once(X: np.ndarray) -> float:
    n, d = X.shape
    if n < 2: return np.nan
    rng = np.random.default_rng()
    U = rng.standard_normal((PFI_DIRS, d)).astype(np.float32)
    U /= np.linalg.norm(U, axis=1, keepdims=True) + 1e-9
    S = U @ X.T
    m = np.max(S, axis=1, keepdims=True)
    logZ = (m + np.log(np.sum(np.exp(S - m), axis=1, keepdims=True))).ravel()
    lo = np.percentile(logZ, PFI_Q_LO)
    hi = np.percentile(logZ, PFI_Q_HI)
    return float(np.exp(lo - hi))  # ≈ min Z / max Z (robust)

def _vmf_kappa_once(X: np.ndarray) -> float:
    if X.shape[0] < 2: return np.nan
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    R = np.linalg.norm(Xn.mean(axis=0))
    d = Xn.shape[1]
    if R < 1e-9: return 0.0
    # standard closed-form approximation
    return float(max(R * (d - R**2) / (1.0 - R**2 + 1e-9), 0.0))

# --- Linear ID (fast) ---
def _pcaXX_once(X: np.ndarray, var_ratio: float) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    c = np.cumsum(lam); thr = c[-1] * var_ratio
    return float(np.searchsorted(c, thr) + 1)

def _pca95_once(X: np.ndarray) -> float:
    return _pcaXX_once(X, 0.95)

def _pca99_once(X: np.ndarray) -> float:
    return _pcaXX_once(X, 0.99)

def _erank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    p = lam / (lam.sum() + EPS)
    H = -(p * np.log(p + EPS)).sum()
    return float(np.exp(H))

def _pr_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    s1 = lam.sum(); s2 = (lam**2).sum()
    return float((s1**2) / (s2 + EPS))

def _stable_rank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0: return np.nan
    return float(lam.sum() / (lam.max() + EPS))

# --- Non-linear (heavy) ---
def _dadapy_twonn_once(X: np.ndarray) -> float:
    if not HAS_DADAPY: return np.nan
    d = Data(coordinates=_jitter_unique(X))
    id_est, _, _ = d.compute_id_2NN()
    return float(id_est)

def _dadapy_gride_once(X: np.ndarray) -> float:
    if not HAS_DADAPY: return np.nan
    d = Data(coordinates=_jitter_unique(X))
    range_max = min(int(DADAPY_GRID_RANGE_MAX), X.shape[0] - 1)
    if range_max < 2:
        return float("nan")
    d.compute_distances(maxk=range_max)
    ids, _, _ = d.return_id_scaling_gride(range_max=range_max)
    return float(ids[-1])

def _skdim_factory(name: str):
    """Return a factory that builds a fresh skdim estimator each call, or None."""
    if not HAS_SKDIM: return None
    mapping = {
        "mom": MOM, "tle": TLE, "corrint": CorrInt, "fishers": FisherS,
        "lpca": lPCA, "lpca99": lPCA,
        "mle": MLE, "danco": DANCo,  "mind_ml": MiND_ML,
        "mada": MADA, "knn": KNN,
    }
    cls = mapping.get(name)
    if cls is None: return None

    def _builder():
        if name == "lpca":      # FO variant
            return cls(ver="FO")
        elif name == "lpca99":  # ratio (0.99) variant
            return cls(ver="ratio", alphaRatio=0.99)
        else:
            return cls()
    return _builder

def _skdim_once_builder(name: str) -> Callable[[np.ndarray], float] | None:
    build = _skdim_factory(name)
    if build is None: return None

    def _once(X: np.ndarray) -> float:
        est = build()
        est.fit(_jitter_unique(X))
        return float(getattr(est, "dimension_", np.nan))
    return _once


# =============================== DATA ===============================
def load_word_df(csv_path: str, exclude_pos: set[str] = EXCLUDE_POS):
    df = pd.read_csv(csv_path, usecols=["sentence_id","tokens","pos"])
    df["sentence_id"] = df["sentence_id"].astype(str)  # keep string IDs
    df.tokens = df.tokens.apply(_to_list); df.pos = df.pos.apply(_to_list)

    rows = []
    for sid, toks, poss in df[["sentence_id","tokens","pos"]].itertuples(index=False):
        for wid, (tok, p) in enumerate(zip(toks, poss)):
            if p not in exclude_pos:
                rows.append((sid, wid, p, tok))
    word_df = pd.DataFrame(rows, columns=["sentence_id","word_id","pos","word"])
    return df, word_df

def sample_raw(word_df: pd.DataFrame, per_pos_cap: int = RAW_MAX_PER_POS) -> pd.DataFrame:
    """Per-POS cap without frequency matching."""
    picks = []
    for p, sub in word_df.groupby("pos", sort=False):
        n = min(len(sub), per_pos_cap)
        picks.append(sub.sample(n, random_state=RAND_SEED, replace=False))
    return pd.concat(picks, ignore_index=True)


def make_class_palette(classes: List[str]) -> Dict[str, Tuple[float, float, float]]:
    """
    Return a deterministic mapping {class -> RGB tuple} with as many distinct
    qualitative colors as needed. Up to ~60 unique colors without reuse.
    """
    # Try three matplotlib tab palettes first (20 + 20 + 20)
    base_colors: List[Tuple[float, float, float]] = []
    for name in ("tab20", "tab20b", "tab20c"):
        try:
            base_colors.extend(sns.color_palette(name, 20))
        except Exception:
            pass

    # If classes exceed our pool, fall back to evenly spaced hues
    if len(base_colors) < len(classes):
        base_colors = list(sns.color_palette("husl", len(classes)))  # evenly spaced hues

    # Deterministic order (sorted) -> stable color assignment
    ordered = list(sorted(classes))
    return {cls: base_colors[i % len(base_colors)] for i, cls in enumerate(ordered)}

# =============================== EMBEDDING ===============================
def embed_subset(df_all_sentences: pd.DataFrame,
                 subset_df: pd.DataFrame,
                 baseline: str = BASELINE,
                 word_rep_mode: str = WORD_REP_MODE,
                 batch_size: int = BATCH_SIZE) -> Tuple[np.ndarray, np.ndarray]:
    """Return reps (L,N,D) and filled mask (N,) for the selected tokens."""
    df_all_sentences["sentence_id"] = df_all_sentences["sentence_id"].astype(str)
    subset_df["sentence_id"] = subset_df["sentence_id"].astype(str)

    # sid -> list[(global_idx, word_id)]
    by_sid: Dict[str, List[Tuple[int,int]]] = {}
    for gidx, (sid, wid) in enumerate(subset_df[["sentence_id","word_id"]].itertuples(index=False)):
        by_sid.setdefault(str(sid), []).append((gidx, int(wid)))

    # Materialize in the exact order we will batch
    sids = list(by_sid.keys())
    df_sel = (df_all_sentences[df_all_sentences.sentence_id.isin(sids)]
              .drop_duplicates("sentence_id")
              .set_index("sentence_id")
              .loc[sids])

    tokzr = AutoTokenizer.from_pretrained(baseline, use_fast=True, add_prefix_space=True)
    tokzr.pad_token = tokzr.eos_token
    enc_kwargs = dict(is_split_into_words=True, return_tensors="pt", padding=True)
    if "add_prefix_space" in inspect.signature(tokzr.__call__).parameters:
        enc_kwargs["add_prefix_space"] = True

    model = AutoModel.from_pretrained(baseline, output_hidden_states=True).eval().to(device)
    if device == "cuda":
        model.half()

    L = model.config.num_hidden_layers + 1
    D = model.config.hidden_size
    N = len(subset_df)

    reps   = np.zeros((L, N, D), np.float16)
    filled = np.zeros(N, dtype=bool)

    with torch.no_grad(), torch.cuda.amp.autocast(device == "cuda"):
        for start in tqdm(range(0, len(sids), batch_size), desc=f"{baseline} (embed subset)"):
            batch_ids    = sids[start : start + batch_size]
            batch_tokens = df_sel.loc[batch_ids, "tokens"].tolist()

            enc_be = tokzr(batch_tokens, **enc_kwargs)
            enc_t  = {k: v.to(device) for k, v in enc_be.items()}
            out = model(**enc_t)
            h = torch.stack(out.hidden_states).detach().cpu().numpy().astype(np.float32)  # (L,B,T,D)

            for b, sid in enumerate(batch_ids):
                # word_id -> token positions for this item
                mp = {}
                for tidx, wid in enumerate(enc_be.word_ids(b)):
                    if wid is not None:
                        mp.setdefault(int(wid), []).append(int(tidx))

                for gidx, wid in by_sid.get(sid, []):
                    toks = mp.get(wid)
                    if not toks: continue
                    if word_rep_mode == "first":
                        vec = h[:, b, toks[0], :]
                    else:
                        vec = h[:, b, toks, :].mean(axis=1)
                    reps[:, gidx, :] = vec.astype(np.float16, copy=False)
                    filled[gidx] = True

            # free batch buffers
            del enc_be, enc_t, out, h
            if device == "cuda": torch.cuda.empty_cache()

    missing = int((~filled).sum())
    if missing:
        print(f"⚠ Missing vectors for {missing} of {N} sampled words")
    del model; gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    return reps, filled


# =============================== BOOTSTRAP CORE ===============================
def _bs_layer_loop(rep_sub: np.ndarray, M: int, n_reps: int, compute_once: Callable[[np.ndarray], float]):
    """Bootstrap: sample M with replacement and apply compute_once(X_layer) -> scalar for each layer."""
    L, N, D = rep_sub.shape
    rng = np.random.default_rng(RAND_SEED)
    A = np.full((n_reps, L), np.nan, np.float32)
    for r in range(n_reps):
        idx = rng.integers(0, N, size=M)
        for l in range(L):
            X = rep_sub[l, idx].astype(np.float32, copy=False)
            try:
                A[r, l] = float(compute_once(X))
            except Exception:
                A[r, l] = np.nan
    mu = np.nanmean(A, axis=0).astype(np.float32)
    lo = np.nanpercentile(A, 2.5, axis=0).astype(np.float32)
    hi = np.nanpercentile(A, 97.5, axis=0).astype(np.float32)
    return mu, lo, hi

# fast metric registry (name -> callable(X)->float)
FAST_ONCE: Dict[str, Callable[[np.ndarray], float]] = {
    "iso": _iso_once,
    "spect": _spect_once,
    "rand": _rand_once,
    "sf": _sf_once,
    "vmf_kappa": _vmf_kappa_once,
    "erank": _erank_once,
    "pr": _pr_once,
    "stable_rank": _stable_rank_once,
}

# heavy metric registry
HEAVY_ONCE: Dict[str, Callable[[np.ndarray], float] | None] = {
    "twonn": _dadapy_twonn_once,
    "gride": _dadapy_gride_once,
    "mom":   _skdim_once_builder("mom"),
    "tle":   _skdim_once_builder("tle"),
    "corrint": _skdim_once_builder("corrint"),
    "fishers": _skdim_once_builder("fishers"),
    "lpca":  _skdim_once_builder("lpca"),
    "lpca95": _skdim_once_builder("lpca95"),
    "lpca99": _skdim_once_builder("lpca99"),
    "mle":   _skdim_once_builder("mle"),
    "mada":  _skdim_once_builder("mada"),
}

LABELS = {
    # Isotropy
    "iso":"IsoScore","spect":"Spectral Ratio","rand":"RandCos |μ|",
    "sf":"Spectral Flatness","vmf_kappa":"vMF κ",
    # Linear ID
    "erank":"Effective Rank","pr":"Participation Ratio","stable_rank":"Stable Rank",
    "lpca95":"lPCA95","lpca99":"lPCA99","lpca":"lPCA FO",
    # Non-linear
    "twonn":"TwoNN ID","gride":"GRIDE",
    "mom":"MOM","tle":"TLE","corrint":"CorrInt",
    "fishers":"FisherS",
    "mle":"MLE","mada":"MADA","knn":"KNN",
}

# Choose plotting order
PLOT_ORDER = (
    "iso","sf","vmf_kappa","spect","rand",
    "erank","pr","stable_rank","lpca95","lpca99","lpca",
    "twonn","gride","mom","tle","corrint","fishers",
    "mle","mada",
)

# metrics you want to compute (you can prune this list to reduce runtime)
#ALL_METRICS = list(PLOT_ORDER)
ALL_METRICS = ["iso"]


# =============================== SAVE / PLOT ===============================
def save_metric_csv_all_pos(metric: str,
                            pos_to_stats: Dict[str, Dict[str, np.ndarray]],
                            layers: np.ndarray,
                            baseline: str,
                            subset_name: str = "raw"):
    rows = []
    for p, stats in pos_to_stats.items():
        mu, lo, hi, n = stats["mean"], stats.get("lo"), stats.get("hi"), stats.get("n", np.nan)
        for l, val in enumerate(mu):
            rows.append({
                "subset": subset_name, "model": baseline, "feature": "pos",
                "class": p, "metric": metric, "layer": int(layers[l]),
                "mean": float(val) if np.isfinite(val) else np.nan,
                "ci_low": float(lo[l]) if isinstance(lo, np.ndarray) and np.isfinite(lo[l]) else np.nan,
                "ci_high": float(hi[l]) if isinstance(hi, np.ndarray) and np.isfinite(hi[l]) else np.nan,
                "n_tokens": int(stats.get("n", 0)), "word_rep_mode": WORD_REP_MODE,
                "source_csv": Path(CSV_PATH).name,
            })
    df = pd.DataFrame(rows)
    out = CSV_DIR / f"pos_{subset_name}_{metric}_{baseline}.csv"
    df.to_csv(out, index=False)

def plot_metric_with_ci(pos_to_stats: Dict[str, Dict[str, np.ndarray]],
                        layers: np.ndarray, metric: str, title: str, out_path: Path,
                        palette: Dict[str, Tuple[float, float, float]] | None = None):
    plt.figure(figsize=(9, 5))
    for p, stats in pos_to_stats.items():
        mu, lo, hi = stats["mean"], stats.get("lo"), stats.get("hi")
        if mu is None or np.all(np.isnan(mu)): 
            continue
        color = palette.get(p) if isinstance(palette, dict) else None
        plt.plot(layers, mu, label=p, lw=1.8, color=color)
        if isinstance(lo, np.ndarray) and isinstance(hi, np.ndarray) and not np.all(np.isnan(lo)):
            plt.fill_between(layers, lo, hi, alpha=0.15, color=color)
    plt.xlabel("Layer"); plt.ylabel(LABELS.get(metric, metric.upper())); plt.title(title)

    # Make the legend compact if there are many classes
    n_classes = len(pos_to_stats)
    ncol = 3 if n_classes > 12 else 2
    plt.legend(ncol=ncol, fontsize="small", title="POS", frameon=False)

    plt.tight_layout(); plt.savefig(out_path, dpi=220); plt.close()



# =============================== DRIVER ===============================
def run_pos_pipeline():
    # 1) Load
    df_all, word_df = load_word_df(CSV_PATH, EXCLUDE_POS)
    POS_TAGS = sorted(word_df.pos.unique())
    palette = make_class_palette(POS_TAGS)
    print(f"✓ corpus ready — {len(word_df):,} tokens across {len(POS_TAGS)} POS")
    print(f"• DADApy: {'available' if HAS_DADAPY else 'missing'}  • scikit-dimension: {'available' if HAS_SKDIM else 'missing'}")

    # 2) Raw sampling only (no frequency match)
    raw_df = sample_raw(word_df, RAW_MAX_PER_POS)
    print("Sample sizes per POS (raw cap):")
    print(raw_df.pos.value_counts().to_dict())

    # 3) Embed once
    reps, filled = embed_subset(df_all, raw_df, BASELINE, WORD_REP_MODE, BATCH_SIZE)
    raw_df = raw_df.reset_index(drop=True).loc[filled].reset_index(drop=True)
    pos_arr = raw_df.pos.values
    L = reps.shape[0]; layers = np.arange(L)
    print(f"✓ embedded {len(raw_df):,} tokens  • layers={L}")

    # 4) Metric-by-metric loop (incremental outputs)
    for metric in ALL_METRICS:
        print(f"\n→ Computing metric: {metric} …")

        # Decide registry & bootstrap settings
        if metric in FAST_ONCE:
            compute_once = FAST_ONCE[metric]
            n_bs = N_BOOTSTRAP_FAST
            Mcap = FAST_BS_MAX_SAMP_PER_POS
        else:
            compute_once = HEAVY_ONCE.get(metric)
            n_bs = N_BOOTSTRAP_HEAVY
            Mcap = HEAVY_BS_MAX_SAMP_PER_POS

        if compute_once is None:
            print(f"  (skipping {metric}: estimator unavailable)")
            continue

        # Per-POS bootstrap
        metric_results: Dict[str, Dict[str, np.ndarray]] = {}
        for p in POS_TAGS:
            idx = np.where(pos_arr == p)[0]
            if idx.size < 3:
                continue
            sub = reps[:, idx]  # (L, n_p, D)
            Np = sub.shape[1]
            M = min(Mcap, Np)

            mu, lo, hi = _bs_layer_loop(sub, M, n_bs, compute_once)
            metric_results[p] = {"mean": mu, "lo": lo, "hi": hi, "n": int(Np)}

        # Save + plot immediately for this metric
        save_metric_csv_all_pos(metric, metric_results, layers, BASELINE, subset_name="raw")
        plot_metric_with_ci(metric_results, layers, metric,
                    title=f"{LABELS.get(metric, metric.upper())} • {BASELINE}",
                    out_path=PLOT_DIR / f"raw_{metric}_{BASELINE}.png",
                    palette=palette)

        print(f"  ✓ saved: CSV= tables/pos_bootstrap/pos_raw_{metric}_{BASELINE}.csv  "
              f"plot= AO_POS/raw_{metric}_{BASELINE}.png")

        # light cleanup for safety
        del metric_results; gc.collect()
        if device == "cuda": torch.cuda.empty_cache()

    # Cleanup
    del reps; gc.collect()
    if device == "cuda": torch.cuda.empty_cache()
    print("\n✓ done (incremental outputs produced per metric).")


if __name__ == "__main__":
    run_pos_pipeline()

In [ ]:


import math
from contextlib import nullcontext

# ---- safety: EPS used by some of your metric fns ----
if "EPS" not in globals():
    EPS = 1e-12

# -------------------- knobs --------------------
N_MAX      = 5_000                 # top subsample size (like Fig A.1)
N_MIN      = 200                    # bottom subsample size
N_POINTS   = 18                     # number of log-spaced points
SEEDS      = (0, 1, 2)              # 3 random seeds (like Fig A.1)
BOOTSTRAP  = True                   # sample with replacement (like Fig A.1)
MAX_LEN    = 512                    # BERT max len
BSZ        = BATCH_SIZE             # use your existing batch size
OUTDIR     = _project_out("convergence"); OUTDIR.mkdir(exist_ok=True, parents=True)

print(f"device={device}  HAS_DADAPY={HAS_DADAPY}  HAS_SKDIM={HAS_SKDIM}  BSZ={BSZ}")

# -------------------- load UD-EWT and build word table (no POS split, but can exclude some POS) --------------------
df_all = pd.read_csv(CSV_PATH, usecols=["sentence_id","tokens","pos"])
df_all["sentence_id"] = df_all["sentence_id"].astype(str)
df_all["tokens"] = df_all["tokens"].apply(_to_list)
df_all["pos"]    = df_all["pos"].apply(_to_list)

rows = []
for sid, toks, poss in df_all[["sentence_id","tokens","pos"]].itertuples(index=False):
    for wid, (tok, p) in enumerate(zip(toks, poss)):
        if p not in EXCLUDE_POS:
            rows.append((sid, wid, tok))
word_df = pd.DataFrame(rows, columns=["sentence_id","word_id","word"])

print(f"Total candidate words (after EXCLUDE_POS): {len(word_df):,}")

# sample up to N_MAX words uniformly (no POS stratification)
N_SAMPLE = min(N_MAX, len(word_df))
subset_df = word_df.sample(n=N_SAMPLE, random_state=RAND_SEED, replace=False).reset_index(drop=True)
print(f"Sampled words: {len(subset_df):,}")

# -------------------- embed LAST LAYER word reps for the sampled words --------------------
def embed_last_layer_words(df_all_sentences: pd.DataFrame,
                           subset_words: pd.DataFrame,
                           baseline: str = BASELINE,
                           word_rep_mode: str = WORD_REP_MODE,
                           batch_size: int = BSZ,
                           max_length: int = MAX_LEN):
    """
    Returns:
      X_last: (N,D) float16 word vectors from the LAST layer
      filled: (N,) boolean mask for successfully aligned words
    """
    df_all_sentences = df_all_sentences.copy()
    subset_words = subset_words.copy()
    df_all_sentences["sentence_id"] = df_all_sentences["sentence_id"].astype(str)
    subset_words["sentence_id"]     = subset_words["sentence_id"].astype(str)

    # sid -> list[(global_idx, word_id)]
    by_sid = {}
    for gidx, (sid, wid) in enumerate(subset_words[["sentence_id","word_id"]].itertuples(index=False)):
        by_sid.setdefault(str(sid), []).append((int(gidx), int(wid)))

    sids = list(by_sid.keys())

    df_sel = (
        df_all_sentences[df_all_sentences.sentence_id.isin(sids)]
        .drop_duplicates("sentence_id")
        .set_index("sentence_id")
        .loc[sids]
    )

    tokzr = AutoTokenizer.from_pretrained(baseline, use_fast=True)
    model = AutoModel.from_pretrained(baseline).eval().to(device)
    if device == "cuda":
        model.half()

    D = model.config.hidden_size
    N = len(subset_words)
    X_last = np.zeros((N, D), dtype=np.float16)
    filled = np.zeros(N, dtype=bool)

    enc_kwargs = dict(
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )

    amp_ctx = torch.cuda.amp.autocast if device == "cuda" else nullcontext

    with torch.no_grad():
        for start in tqdm(range(0, len(sids), batch_size), desc=f"{baseline} last-layer word reps"):
            batch_ids    = sids[start:start+batch_size]
            batch_tokens = df_sel.loc[batch_ids, "tokens"].tolist()

            enc = tokzr(batch_tokens, **enc_kwargs)
            enc_t = {k: v.to(device) for k, v in enc.items()}

            with amp_ctx():
                out = model(**enc_t)
                h_last = out.last_hidden_state.detach().cpu().numpy().astype(np.float32)  # (B,T,D)

            for b, sid in enumerate(batch_ids):
                mp = {}
                for tidx, wid in enumerate(enc.word_ids(b)):
                    if wid is not None:
                        mp.setdefault(int(wid), []).append(int(tidx))

                for gidx, wid in by_sid.get(sid, []):
                    toks = mp.get(wid)
                    if not toks:
                        continue
                    if word_rep_mode == "first":
                        vec = h_last[b, toks[0], :]
                    else:
                        vec = h_last[b, toks, :].mean(axis=0)
                    X_last[gidx] = vec.astype(np.float16, copy=False)
                    filled[gidx] = True

            del enc, enc_t, out, h_last
            if device == "cuda":
                torch.cuda.empty_cache()

    del model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    return X_last, filled

X_last, filled = embed_last_layer_words(df_all, subset_df)
X = X_last[filled].astype(np.float32, copy=False)
print(f"Embedded + aligned words: {X.shape[0]:,}  dim={X.shape[1]}  (missing {int((~filled).sum()):,})")

# -------------------- log-spaced subsample sizes --------------------
def log_sizes(n_min: int, n_max: int, n_points: int) -> np.ndarray:
    n_min = max(2, int(n_min))
    n_max = max(n_min, int(n_max))
    sizes = np.unique(np.round(np.logspace(np.log10(n_min), np.log10(n_max), n_points)).astype(int))
    return sizes

N_TOP = min(N_MAX, X.shape[0])
sizes = log_sizes(N_MIN, N_TOP, N_POINTS)
print("sizes:", sizes[:5], "...", sizes[-5:], f"(count={len(sizes)})")

# -------------------- estimators (match Fig A.1 set) --------------------
# PCA in that legend: use your PCA99 explained-variance ID
def PCA_once(Xsub: np.ndarray) -> float:
    return float(_pca99_once(Xsub))

# skdim helpers
def _skdim_once(cls):
    def _fn(Xsub: np.ndarray) -> float:
        est = cls()
        est.fit(_jitter_unique(Xsub))
        return float(getattr(est, "dimension_", np.nan))
    return _fn

ID_METRICS = {
    "PCA": PCA_once,
}

# dadapy (TwoNN)
if HAS_DADAPY:
    ID_METRICS["TwoNN"] = _dadapy_twonn_once

# skdim estimators
if HAS_SKDIM:
    for nm, label in [
        ("fishers", "FisherS"),
        #("mle",     "MLE"),
       # ("corrint", "CorrInt"),
      #  ("mom",     "MOM"),
     #   ("tle",     "TLE"),
      #  ("mada",    "MADA"),
    ]:
        fn = _skdim_once_builder(nm)
        if fn is not None:
            ID_METRICS[label] = fn

""" 
    try:
        ID_METRICS["ESS"] = _skdim_once(ESS)
    except Exception:
        pass
"""
# IsoScore convergence separately (different scale)
ISO_METRICS = {"IsoScore": _iso_once}

print("ID metrics:", list(ID_METRICS.keys()))
print("ISO metrics:", list(ISO_METRICS.keys()))

# -------------------- precompute bootstrap indices (shared across metrics for fairness/speed) --------------------
N = X.shape[0]
idx_cache = {}  # (n, seed) -> indices
for n in sizes:
    for s in SEEDS:
        rng = np.random.default_rng(int(s))
        idx_cache[(int(n), int(s))] = rng.choice(N, size=int(n), replace=BOOTSTRAP)

# -------------------- convergence compute --------------------
def convergence_df(X: np.ndarray, metrics: dict[str, Callable[[np.ndarray], float]]) -> pd.DataFrame:
    rows = []
    for name, fn in metrics.items():
        print(f"\n→ metric: {name}")
        for n in tqdm(sizes, desc=f"{name}"):
            vals = []
            for s in SEEDS:
                idx = idx_cache[(int(n), int(s))]
                try:
                    vals.append(float(fn(X[idx])))
                except Exception as e:
                    vals.append(np.nan)
            vals = np.asarray(vals, dtype=np.float64)
            rows.append({
                "metric": name,
                "N": int(n),
                "mean": float(np.nanmean(vals)),
                "std": float(np.nanstd(vals, ddof=1)),
            })
    return pd.DataFrame(rows)

df_id  = convergence_df(X, ID_METRICS)
df_iso = convergence_df(X, ISO_METRICS)

df_id.to_csv(OUTDIR / "convergence_ID.csv", index=False)
df_iso.to_csv(OUTDIR / "convergence_IsoScore.csv", index=False)
print("saved:", OUTDIR / "convergence_ID.csv")
print("saved:", OUTDIR / "convergence_IsoScore.csv")

# -------------------- plots (match Fig A.1 style: mean ± 1 std, x-axis in 1e4 units) --------------------
def plot_convergence(df: pd.DataFrame, title: str, ylabel: str, outpath: Path):
    plt.figure(figsize=(7.4, 4.9))
    for name, g in df.groupby("metric", sort=False):
        g = g.sort_values("N")
        x = g["N"].to_numpy() / 1e4
        y = g["mean"].to_numpy()
        s = g["std"].to_numpy()
        plt.plot(x, y, lw=2, label=name)
        plt.fill_between(x, y - s, y + s, alpha=0.15)
    plt.xlabel("N Samples (1e4)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(frameon=False, fontsize="small", ncol=2)
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.show()

plot_convergence(
    df_id,
    title=f"Convergence of ID Methods (UD-EWT sample, {BASELINE} last layer)",
    ylabel="ID Estimate",
    outpath=OUTDIR / "convergence_ID.png",
)

plot_convergence(
    df_iso,
    title=f"Convergence of IsoScore (UD-EWT sample, {BASELINE} last layer)",
    ylabel="IsoScore",
    outpath=OUTDIR / "convergence_IsoScore.png",
)


In [ ]:
import math
from contextlib import nullcontext
from pathlib import Path

# ---- safety: EPS used by some of your metric fns ----
if "EPS" not in globals():
    EPS = 1e-12

# ---- ensure DADApy GRIDE range exists ----
if "DADAPY_GRID_RANGE_MAX" not in globals():
    DADAPY_GRID_RANGE_MAX = 64  # typical; docs show range_max=64 :contentReference[oaicite:3]{index=3}

# -------------------- knobs --------------------
N_MAX      = 10_000                 # top subsample size (like Fig A.1)
N_MIN      = 200                    # bottom subsample size
N_POINTS   = 18                     # number of log-spaced points
SEEDS      = (0, 1, 2)              # 3 random seeds
BOOTSTRAP  = True                   # sample with replacement
MAX_LEN    = 512                    # BERT max len
BSZ        = BATCH_SIZE             # use your existing batch size
OUTDIR     = _project_out("convergence"); OUTDIR.mkdir(exist_ok=True, parents=True)

print(f"device={device}  HAS_DADAPY={HAS_DADAPY}  HAS_SKDIM={HAS_SKDIM}  BSZ={BSZ}")

# -------------------- load UD-EWT and build word table (no POS split, but can exclude some POS) --------------------
df_all = pd.read_csv(CSV_PATH, usecols=["sentence_id","tokens","pos"])
df_all["sentence_id"] = df_all["sentence_id"].astype(str)
df_all["tokens"] = df_all["tokens"].apply(_to_list)
df_all["pos"]    = df_all["pos"].apply(_to_list)

rows = []
for sid, toks, poss in df_all[["sentence_id","tokens","pos"]].itertuples(index=False):
    for wid, (tok, p) in enumerate(zip(toks, poss)):
        if p not in EXCLUDE_POS:
            rows.append((sid, wid, tok))
word_df = pd.DataFrame(rows, columns=["sentence_id","word_id","word"])
del rows

print(f"Total candidate words (after EXCLUDE_POS): {len(word_df):,}")

# sample up to N_MAX words uniformly (no POS stratification)
N_SAMPLE = min(N_MAX, len(word_df))
subset_df = word_df.sample(n=N_SAMPLE, random_state=RAND_SEED, replace=False).reset_index(drop=True)
print(f"Sampled words: {len(subset_df):,}")

# -------------------- embed LAST LAYER word reps for the sampled words --------------------
def embed_last_layer_words(df_all_sentences: pd.DataFrame,
                           subset_words: pd.DataFrame,
                           baseline: str = BASELINE,
                           word_rep_mode: str = WORD_REP_MODE,
                           batch_size: int = BSZ,
                           max_length: int = MAX_LEN):
    """
    Returns:
      X_last: (N,D) float16 word vectors from the LAST layer
      filled: (N,) boolean mask for successfully aligned words
    """
    # IMPORTANT (low RAM): avoid .copy() here; df_all already prepared above.

    # sid -> list[(global_idx, word_id)]
    by_sid = {}
    for gidx, (sid, wid) in enumerate(subset_words[["sentence_id","word_id"]].itertuples(index=False)):
        by_sid.setdefault(str(sid), []).append((int(gidx), int(wid)))

    sids = list(by_sid.keys())
    df_sel = (
        df_all_sentences[df_all_sentences.sentence_id.isin(sids)]
        .drop_duplicates("sentence_id")
        .set_index("sentence_id")
        .loc[sids]
    )

    tokzr = AutoTokenizer.from_pretrained(baseline, use_fast=True, add_prefix_space=True)
    tokzr.pad_token = tokzr.eos_token
    model = AutoModel.from_pretrained(baseline).eval().to(device)
    if device == "cuda":
        model.half()

    D = model.config.hidden_size
    N = len(subset_words)
    X_last = np.zeros((N, D), dtype=np.float16)
    filled = np.zeros(N, dtype=bool)

    enc_kwargs = dict(
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )

    amp_ctx = torch.cuda.amp.autocast if device == "cuda" else nullcontext

    with torch.no_grad():
        for start in tqdm(range(0, len(sids), batch_size), desc=f"{baseline} last-layer word reps"):
            batch_ids    = sids[start:start+batch_size]
            batch_tokens = df_sel.loc[batch_ids, "tokens"].tolist()

            enc = tokzr(batch_tokens, **enc_kwargs)
            enc_t = {k: v.to(device) for k, v in enc.items()}

            with amp_ctx():
                out = model(**enc_t)
                h_last = out.last_hidden_state.detach().cpu().numpy().astype(np.float32)  # (B,T,D)

            for b, sid in enumerate(batch_ids):
                mp = {}
                for tidx, wid in enumerate(enc.word_ids(b)):
                    if wid is not None:
                        mp.setdefault(int(wid), []).append(int(tidx))

                for gidx, wid in by_sid.get(sid, []):
                    toks = mp.get(wid)
                    if not toks:
                        continue
                    if word_rep_mode == "first":
                        vec = h_last[b, toks[0], :]
                    else:
                        vec = h_last[b, toks, :].mean(axis=0)
                    X_last[gidx] = vec.astype(np.float16, copy=False)
                    filled[gidx] = True

            del enc, enc_t, out, h_last
            if device == "cuda":
                torch.cuda.empty_cache()

    del model
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    return X_last, filled

# -------------------- run embed --------------------
X_last, filled = embed_last_layer_words(df_all, subset_df)

# RAM-LOW: keep base matrix in float16, NOT float32
X = X_last[filled]  # float16
del X_last
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

print(f"Embedded + aligned words: {X.shape[0]:,}  dim={X.shape[1]}  dtype={X.dtype}  (missing {int((~filled).sum()):,})")

# Optional RAM-LOW: free big dataframes once embedding is done
del df_all, word_df, subset_df
gc.collect()

# -------------------- log-spaced subsample sizes --------------------
def log_sizes(n_min: int, n_max: int, n_points: int) -> np.ndarray:
    n_min = max(2, int(n_min))
    n_max = max(n_min, int(n_max))
    return np.unique(np.round(np.logspace(np.log10(n_min), np.log10(n_max), n_points)).astype(int))

N_TOP = min(N_MAX, X.shape[0])
sizes = log_sizes(N_MIN, N_TOP, N_POINTS)
print("sizes:", sizes[:5], "...", sizes[-5:], f"(count={len(sizes)})")

# -------------------- spectral bundle (ONE SVD per sample) --------------------
# Includes Local PCA (FO): eigenvalues > alphaFO * max_eig, default alphaFO=0.05 per lPCA docs :contentReference[oaicite:4]{index=4}
ALPHA_FO = 0.05  # match skdim.id.lPCA default alphaFO

def _spectral_bundle_from_lam(lam: np.ndarray, var_ratio: float = 0.99, alphaFO: float = ALPHA_FO) -> dict[str, float]:
    if lam.size == 0 or not np.isfinite(lam).all() or lam.max() <= 0:
        return {
            "Spectral Flatness": np.nan,
            "Effective Rank": np.nan,
            "Participation Ratio": np.nan,
            "Stable Rank": np.nan,
            "PCA": np.nan,
            "Local PCA (lPCA FO)": np.nan,
        }

    lam = lam.astype(np.float64, copy=False)

    gm = float(np.exp(np.mean(np.log(lam + EPS))))
    am = float(lam.mean() + EPS)
    sf = float(gm / am)

    p = lam / (lam.sum() + EPS)
    H = float(-(p * np.log(p + EPS)).sum())
    erank = float(np.exp(H))

    s1 = float(lam.sum())
    s2 = float((lam**2).sum())
    pr = float((s1*s1) / (s2 + EPS))

    stable = float(s1 / (lam.max() + EPS))

    c = np.cumsum(lam)
    thr = c[-1] * float(var_ratio)
    pca = float(np.searchsorted(c, thr) + 1)

    lpca_fo = float(np.sum(lam > (alphaFO * lam.max())))

    return {
        "Spectral Flatness": sf,
        "Effective Rank": erank,
        "Participation Ratio": pr,
        "Stable Rank": stable,
        "PCA": pca,
        "Local PCA (lPCA FO)": lpca_fo,
    }

# -------------------- low-mem wrappers for GRIDE and ESS --------------------
# GRIDE usage per DADApy docs: compute_distances(maxk=...) then return_id_scaling_gride(range_max=...) :contentReference[oaicite:5]{index=5}
def GRIDE_once(Xsub: np.ndarray) -> float:
    if not HAS_DADAPY:
        return np.nan
    try:
        d = Data(coordinates=_jitter_unique(Xsub))
        range_max = min(int(DADAPY_GRID_RANGE_MAX), X.shape[0] - 1)
        if range_max < 2:
            return float("nan")
        d.compute_distances(maxk=range_max)
        ids, _, _ = d.return_id_scaling_gride(range_max=range_max)
        out = float(ids[-1])
        del d
        return out
    except Exception:
        return np.nan

def TwoNN_once(Xsub: np.ndarray) -> float:
    if not HAS_DADAPY:
        return np.nan
    try:
        d = Data(coordinates=_jitter_unique(Xsub))
        id_est, _, _ = d.compute_id_2NN()
        out = float(id_est)
        del d
        return out
    except Exception:
        return np.nan

# ESS per scikit-dimension docs: fit() then read dimension_ :contentReference[oaicite:6]{index=6}
def ESS_once(Xsub: np.ndarray) -> float:
    if not HAS_SKDIM:
        return np.nan
    try:
        est = ESS()
        est.fit(_jitter_unique(Xsub))
        out = float(getattr(est, "dimension_", np.nan))
        del est
        return out
    except Exception:
        return np.nan

# -------------------- metrics you will output --------------------
ISO_NAMES = ["IsoScore", "Spectral Flatness", "vMF κ"]

ID_NAMES = [
    "PCA",
    "Effective Rank",
    "Participation Ratio",
    "Stable Rank",
    "Local PCA (lPCA FO)",
]

# Extra estimators (include GRIDE + ESS here)
EXTRA_ID = {}

if HAS_DADAPY:
    EXTRA_ID["TwoNN"] = TwoNN_once
    EXTRA_ID["GRIDE"] = GRIDE_once

if HAS_SKDIM:
    # your skdim estimators (through builder)
    for nm, label in [
        ("fishers", "FisherS"),
        ("mle", "MLE"),
        ("corrint", "CorrInt"),
        ("mom", "MOM"),
        ("tle", "TLE"),
        ("mada", "MADA"),
    ]:
        fn = _skdim_once_builder(nm)
        if fn is not None:
            EXTRA_ID[label] = fn

    # ESS explicitly (works even if your factory doesn't map "ess")
    EXTRA_ID["ESS"] = ESS_once

print("Extra ID metrics:", list(EXTRA_ID.keys()))

# -------------------- deterministic on-the-fly bootstrap indices (RAM-LOW) --------------------
def _sample_idx(N: int, n: int, seed: int) -> np.ndarray:
    ss = np.random.SeedSequence([int(RAND_SEED), int(seed), int(n)])
    rng = np.random.default_rng(ss)
    return rng.choice(N, size=int(n), replace=BOOTSTRAP)

# -------------------- convergence (compute ALL metrics per subsample) --------------------
N = X.shape[0]
rows_id = []
rows_iso = []

for n in tqdm(sizes, desc="convergence"):
    id_vals  = {k: [] for k in ID_NAMES}
    iso_vals = {k: [] for k in ISO_NAMES}
    extra_id_vals = {k: [] for k in EXTRA_ID.keys()}

    for s in SEEDS:
        idx = _sample_idx(N, int(n), int(s))

        # one float32 subsample per (n,seed)
        Xsub = X[idx].astype(np.float32, copy=False)

        # spectrum once
        lam = _eigvals_from_X(Xsub)
        bundle = _spectral_bundle_from_lam(lam, var_ratio=0.99, alphaFO=ALPHA_FO)

        # isotropy metrics
        iso_vals["IsoScore"].append(float(_iso_once(Xsub)))
        iso_vals["vMF κ"].append(float(_vmf_kappa_once(Xsub)))
        iso_vals["Spectral Flatness"].append(float(bundle["Spectral Flatness"]))

        # ID/spectrum metrics
        id_vals["PCA"].append(float(bundle["PCA"]))
        id_vals["Effective Rank"].append(float(bundle["Effective Rank"]))
        id_vals["Participation Ratio"].append(float(bundle["Participation Ratio"]))
        id_vals["Stable Rank"].append(float(bundle["Stable Rank"]))
        id_vals["Local PCA (lPCA FO)"].append(float(bundle["Local PCA (lPCA FO)"]))

        # extra ID estimators (TwoNN/GRIDE/ESS/etc.)
        for name, fn in EXTRA_ID.items():
            try:
                extra_id_vals[name].append(float(fn(Xsub)))
            except Exception:
                extra_id_vals[name].append(np.nan)

        # cleanup per seed
        del idx, Xsub, lam, bundle
        if device == "cuda":
            torch.cuda.empty_cache()

    # aggregate mean/std over seeds
    for m in ID_NAMES:
        vals = np.asarray(id_vals[m], dtype=np.float64)
        rows_id.append({"metric": m, "N": int(n),
                        "mean": float(np.nanmean(vals)),
                        "std":  float(np.nanstd(vals, ddof=1))})

    for m in ISO_NAMES:
        vals = np.asarray(iso_vals[m], dtype=np.float64)
        rows_iso.append({"metric": m, "N": int(n),
                         "mean": float(np.nanmean(vals)),
                         "std":  float(np.nanstd(vals, ddof=1))})

    for m in EXTRA_ID.keys():
        vals = np.asarray(extra_id_vals[m], dtype=np.float64)
        rows_id.append({"metric": m, "N": int(n),
                        "mean": float(np.nanmean(vals)),
                        "std":  float(np.nanstd(vals, ddof=1))})

df_id  = pd.DataFrame(rows_id)
df_iso = pd.DataFrame(rows_iso)

df_id.to_csv(OUTDIR / "convergence_ID1gpt.csv", index=False)
df_iso.to_csv(OUTDIR / "convergence_IsoScore1gpt.csv", index=False)
print("saved:", OUTDIR / "convergence_ID1gpt.csv")
print("saved:", OUTDIR / "convergence_IsoScore1gpt.csv")

# -------------------- plots --------------------
def plot_convergence(df: pd.DataFrame, title: str, ylabel: str, outpath: Path):
    plt.figure(figsize=(7.4, 4.9))
    for name, g in df.groupby("metric", sort=False):
        g = g.sort_values("N")
        x = g["N"].to_numpy() / 1e4
        y = g["mean"].to_numpy()
        s = g["std"].to_numpy()
        plt.plot(x, y, lw=2, label=name)
        plt.fill_between(x, y - s, y + s, alpha=0.15)
    plt.xlabel("N Samples (1e4)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(frameon=False, fontsize="small", ncol=2)
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.show()

plot_convergence(
    df_id,
    title=f"Convergence of ID Methods (UD-EWT sample, {BASELINE} last layer)",
    ylabel="ID Estimate",
    outpath=OUTDIR / "convergence_ID1.png",
)

plot_convergence(
    df_iso,
    title=f"Convergence of Isotropy Metrics (UD-EWT sample, {BASELINE} last layer)",
    ylabel="Estimate",
    outpath=OUTDIR / "convergence_IsoScore1gpt.png",
)


In [ ]:
# ===================== NICE PLOTS (small, paper-ready, vertical stack) =====================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- inputs (auto-detect) ----------
def _pick(*cands):
    for c in cands:
        p = Path(c)
        if p.exists():
            return p
    raise FileNotFoundError(f"None of these exist: {cands}")

ID_CSV  = _pick("convergence_ID1gpt.csv", "convergence_ID1gpt.csv", "convergence/convergence_ID1gpt.csv")
ISO_CSV = _pick("convergence_IsoScore1gpt.csv", "convergence_IsoScore1gpt.csv", "convergence/convergence_IsoScore1gpt.csv")

df_id  = pd.read_csv(ID_CSV)
df_id = df_id[df_id["metric"] != "PCA"].copy()
df_id = df_id[df_id["metric"] != "MADA"].copy()

df_iso = pd.read_csv(ISO_CSV)

# ---------- hygiene ----------
for df in (df_id, df_iso):
    df["N"]    = pd.to_numeric(df["N"], errors="coerce").astype(int)
    df["mean"] = pd.to_numeric(df["mean"], errors="coerce")
    df["std"]  = pd.to_numeric(df["std"], errors="coerce").fillna(0.0)

# ---------- rename metrics ----------
RENAME = {
    "PCA": "PCA@99",
    "PCA (99%)": "PCA@99",
    "pca99": "PCA@99",
    "pca@99": "PCA@99",
    "Local PCA (lPCA FO)": "PCA FO",
    "lPCA FO": "PCA FO",
    "lPCA (FO)": "PCA FO",
    "lpca": "PCA FO",
}

df_id["metric"] = df_id["metric"].replace(RENAME)

OUT = _project_out("convergence_plots")
OUT.mkdir(exist_ok=True, parents=True)

# ---------- metric grouping (markers/linestyles) ----------
LINEAR_ID = {
  #  "PCA@99",
    "PCA FO",
    "Effective Rank",
    "Participation Ratio",
    "Stable Rank",
}
NONLINEAR_ID = {
    "TwoNN", "GRIDE", "FisherS", "MLE", "CorrInt", "MOM", "TLE", "MADA", "ESS", "KNN"
}

def _ordered_metrics(df, preferred):
    present = list(df["metric"].unique())
    out = [m for m in preferred if m in present]
    out += [m for m in present if m not in out]
    return out

ID_ORDER = _ordered_metrics(
    df_id,
    [
      #  "PCA@99",
        "PCA FO",
        "Effective Rank", "Participation Ratio", "Stable Rank",
        "TwoNN", "GRIDE",
        "FisherS", "MLE", "CorrInt", "MOM", "TLE", "MADA", "ESS", "KNN"
    ],
)
ISO_ORDER = _ordered_metrics(df_iso, ["IsoScore", "Spectral Flatness", "vMF κ"])

def _style(metric_name: str):
    if metric_name in LINEAR_ID:
        return dict(marker="o", linestyle="-")
    if metric_name in NONLINEAR_ID:
        return dict(marker="s", linestyle="--")
    return dict(marker="^", linestyle="-")

# ---------- function to plot single axes ----------
def plot_convergence_ax(ax, df, metric_order, ylabel, title,
                        x_scale=1e4, legend=True, legend_ncol=2):

    for metric in metric_order:
        g = df[df["metric"] == metric].sort_values("N")
        if g.empty:
            continue

        x = (g["N"].to_numpy(dtype=float) / x_scale)
        y = g["mean"].to_numpy(dtype=float)
        s = g["std"].to_numpy(dtype=float)
        markevery = max(1, len(x) // 7)

        st = _style(metric)
        (line,) = ax.plot(
            x, y,
            lw=1.6,
            label=metric,
            marker=st["marker"],
            markersize=3.2,
            markevery=markevery,
            linestyle=st["linestyle"],
        )
        c = line.get_color()
        ax.fill_between(x, y - s, y + s, alpha=0.15, color=c)

    ax.set_xlabel("N Samples (1e4)", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.25)
    ax.tick_params(axis="both", labelsize=10)

    if legend:
        ax.legend(
            frameon=False,
            fontsize=7.2,
            ncol=legend_ncol,
            handlelength=2.0,
            columnspacing=0.8,
            borderaxespad=0.2,
            loc="best",
        )

# ---------- separate small figures (unchanged) ----------
fig1, ax1 = plt.subplots(figsize=(3.55, 2.65))
plot_convergence_ax(
    ax1, df_id, ID_ORDER,
    ylabel="ID estimate",
    title="Intrinsic dimensionality (convergence)",
    legend=True,
    legend_ncol=2,
)
fig1.tight_layout(pad=0.3)
fig1.savefig(OUT / "convergence_ID_small.pdf", bbox_inches="tight")
fig1.savefig(OUT / "convergence_ID_small.png", dpi=300, bbox_inches="tight")
plt.show()

fig2, ax2 = plt.subplots(figsize=(3.55, 2.65))
plot_convergence_ax(
    ax2, df_iso, ISO_ORDER,
    ylabel="Estimate",
    title="Isotropy metrics (convergence)",
    legend=True,
    legend_ncol=1,
)
fig2.tight_layout(pad=0.3)
fig2.savefig(OUT / "convergence_Iso_small.pdf", bbox_inches="tight")
fig2.savefig(OUT / "convergence_Iso_small.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved:")
print(" ", OUT / "convergence_ID_small.pdf")
print(" ", OUT / "convergence_Iso_small.pdf")

# ---------- stacked figure: vertical layout 2x1 ----------
fig, (ax_top, ax_bot) = plt.subplots(
    2, 1,
    figsize=(4.5, 6.0),   # taller figure, two rows
    gridspec_kw=dict(hspace=0.4),
)

plot_convergence_ax(
    ax_top, df_id, ID_ORDER,
    ylabel="ID estimate",
    title="ID",
    legend=False,
)
plot_convergence_ax(
    ax_bot, df_iso, ISO_ORDER,
    ylabel="Estimate",
    title="Isotropy metrics",
    legend=False,
)

# shared legend at bottom
handles_top, labels_top = ax_top.get_legend_handles_labels()
handles_bot, labels_bot = ax_bot.get_legend_handles_labels()
handles = handles_top + handles_bot
labels  = labels_top  + labels_bot

fig.legend(
    handles, labels,
    frameon=False,
    fontsize=12,
    ncol=3,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.25),
    handlelength=2.0,
    columnspacing=0.9,
)

fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(OUT / "convergence_vertical.pdf", bbox_inches="tight")
fig.savefig(OUT / "convergence_vertical.png", dpi=300, bbox_inches="tight")
plt.show()

print("Also saved:")
print(" ", OUT / "convergence_vertical.pdf")
